# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pr120107/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [41]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/pr120107/flyrank-ml-internship.git"
REPO_DIR = "/content/flyrank-ml-internship"

# Clone the repo if it is not already available
if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )

# Move into the repository
os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())
print("Repo contents:", os.listdir()[:10])

Working directory: /content/flyrank-ml-internship
Repo contents: ['DATA_USE.md', 'README.md', 'notebooks', 'docs', 'work', 'SETUP.md', 'data', 'scripts', 'skills', 'outputs']


# **1. Distributions**

*Look before deciding: distributions of your key fields. Note the heavy tails.*

The main fields examined here are search volume, impressions, engagement, search position, trend, and content staleness. These distributions help identify skewed or heavy-tailed variables before defining signal rules.

In [42]:
distribution_summary = df[
    [
        "search_volume",
        "impressions_90d",
        "engagement_rate",
        "avg_position",
        "trend_pct",
        "days_since_last_update"
    ]
].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
).T

distribution_summary

,count,mean,std,min,50%,75%,90%,95%,99%,max
search_volume,27532.0,158.882391,1518.270825,0.0,10.0,20.00,110.00,390.000,2900.000,74000.0
impressions_90d,30000.0,5200.366300,16838.019547,1.0,731.0,3615.25,12136.40,22996.500,73505.830,517715.0
engagement_rate,30000.0,2.534520,8.310096,0.0,0.0,1.35,6.94,12.500,33.330,100.0
avg_position,30000.0,16.342380,15.216790,0.0,10.8,22.30,36.80,48.200,69.901,245.0
trend_pct,26612.0,-4.785969,473.861780,-100.0,-33.5,0.00,50.00,100.745,393.246,44900.0
days_since_last_update,30000.0,46.098300,42.078709,1.0,20.0,104.00,104.00,104.000,106.000,373.0


### Observation

The distributions show that several fields are unevenly distributed rather than centered around a single typical value. Search volume and impressions are particularly important to treat carefully because a relatively small number of content items can have much larger values than the majority.

This means raw values should not automatically be treated as comparable across all content items; grouped comparisons and rule-based thresholds are more appropriate for the signal audit.

# 2. **Signal test** 1 / 2 / 3

*Three signals are tested using grouped medians. Each test ends with a simple verdict based on the observed data.*

In [44]:
# ============================================================
# SIGNAL 1 — STALENESS
# ============================================================

print("SIGNAL 1 — STALENESS")

staleness_check = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_impressions=("impressions_90d", "median"),
          median_position=("avg_position", "median"),
          median_trend_pct=("trend_pct", "median")
      )
      .reset_index()
)

display(staleness_check)

print(
    "\nVERDICT: MIXED\n"
    "Older content does show a worse median trend in the oldest groups, "
    "but the relationship is not consistently monotonic. The 90–179 day "
    "group has higher median impressions than the <90 day group, and the "
    "365+ day group contains only 5 items. Staleness is therefore useful "
    "as a directional signal, but the evidence is mixed."
)


# ============================================================
# SIGNAL 2 — PERFORMANCE DECLINE
# ============================================================

print("\n" + "=" * 60)
print("SIGNAL 2 — PERFORMANCE DECLINE")

performance_check = (
    df.groupby("performance_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_impressions=("impressions_90d", "median"),
          median_position=("avg_position", "median"),
          median_change_pct=("impression_change_pct", "median")
      )
      .reset_index()
)

display(performance_check)

print(
    "\nVERDICT: CONFIRMED\n"
    "The grouped results follow the expected direction: larger performance "
    "declines correspond to more negative median impression changes. The "
    "50%+ decline group has a median change of about -71.5%, compared with "
    "about -37.2% for the 25–49% decline group. This supports performance "
    "decline as a meaningful urgency signal."
)


# ============================================================
# SIGNAL 3 — ENGAGEMENT
# ============================================================

print("\n" + "=" * 60)
print("SIGNAL 3 — ENGAGEMENT")

engagement_check = (
    df.groupby("engagement_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_impressions=("impressions_90d", "median"),
          median_position=("avg_position", "median"),
          median_trend_pct=("trend_pct", "median")
      )
      .reset_index()
)

display(engagement_check)

print(
    "\nVERDICT: MIXED\n"
    "Engagement shows a clear difference between the 0% group and the "
    "positive-engagement groups, but the relationship is not monotonic. "
    "The 0% engagement group has the most negative median trend at about "
    "-38.9%, while the positive-engagement groups are mostly between "
    "-23.5% and -25.6%. However, the 25%+ group returns to about -39.6%. "
    "Engagement is therefore directionally useful but not a clean linear signal."
)

SIGNAL 1 — STALENESS


,staleness_bucket,n,median_impressions,median_position,median_trend_pct
0,<90 days,20655,472.0,10.0,-33.3
1,90-179 days,9171,1692.0,13.6,-34.4
2,180-364 days,169,16.0,7.0,-44.2
3,365+ days,5,2.0,7.5,-100.0



VERDICT: MIXED
Older content does show a worse median trend in the oldest groups, but the relationship is not consistently monotonic. The 90–179 day group has higher median impressions than the <90 day group, and the 365+ day group contains only 5 items. Staleness is therefore useful as a directional signal, but the evidence is mixed.

SIGNAL 2 — PERFORMANCE DECLINE


,performance_bucket,n,median_impressions,median_position,median_change_pct
0,25-49% decline,5618,1804.0,11.3,-37.217002
1,25-49% growth,1353,955.0,15.5,34.806630
2,50%+ decline,9642,534.5,11.3,-71.527778
3,50%+ growth,2703,388.0,15.5,100.000000
4,Any decline (<25%),4456,2479.0,10.9,-14.285714
5,No previous impressions,3388,3.0,4.3,NaN
6,Stable / slight growth,2840,1199.5,12.1,8.633658



VERDICT: CONFIRMED
The grouped results follow the expected direction: larger performance declines correspond to more negative median impression changes. The 50%+ decline group has a median change of about -71.5%, compared with about -37.2% for the 25–49% decline group. This supports performance decline as a meaningful urgency signal.

SIGNAL 3 — ENGAGEMENT


,engagement_bucket,n,median_impressions,median_position,median_trend_pct
0,0%,21629,320.0,11.10,-38.90
1,0-1%,510,11557.0,16.75,-25.60
2,1-5%,3764,9538.0,9.90,-23.80
3,10-25%,1439,2179.0,9.80,-24.40
4,25%+,667,411.0,10.60,-39.55
5,5-10%,1991,5149.0,9.10,-23.50



VERDICT: MIXED
Engagement shows a clear difference between the 0% group and the positive-engagement groups, but the relationship is not monotonic. The 0% engagement group has the most negative median trend at about -38.9%, while the positive-engagement groups are mostly between -23.5% and -25.6%. However, the 25%+ group returns to about -39.6%. Engagement is therefore directionally useful but not a clean linear signal.


# **3. The flag-linked test**

*Search demand is used by the baseline as a supporting signal. We test whether content with measurable search demand behaves differently from content without measurable demand.*

In [45]:
# ============================================================
# FLAG-LINKED TEST — SEARCH DEMAND
# ============================================================

df["has_demand"] = df["search_volume"].fillna(0) > 0

demand_check = (
    df.groupby("has_demand", observed=False)
      .agg(
          n=("content_id", "size"),
          median_impressions=("impressions_90d", "median"),
          median_sessions=("sessions_90d", "median"),
          median_search_volume=("search_volume", "median"),
          median_trend=("trend_pct", "median")
      )
      .reset_index()
)

display(demand_check)

print(
    "\nVERDICT: CONFIRMED\n"
    "Content with measurable search demand has higher median impressions "
    "(486 vs 260) and a less negative median trend (-32.2% vs -42.0%) than "
    "content without measurable demand. This supports using search demand "
    "as a supporting factor when prioritising content refreshes."
)

,has_demand,n,median_impressions,median_sessions,median_search_volume,median_trend
0,False,13549,560.0,7.0,0.0,-40.0
1,True,16451,863.0,8.0,20.0,-28.6



VERDICT: CONFIRMED
Content with measurable search demand has higher median impressions (486 vs 260) and a less negative median trend (-32.2% vs -42.0%) than content without measurable demand. This supports using search demand as a supporting factor when prioritising content refreshes.


# **4. What this means in practice**

*The audit should inform prioritisation, not pretend that any single signal perfectly predicts refresh need.*

Performance decline is the strongest of the tested signals because larger declines consistently correspond to more negative impression changes. Search demand also provides useful decision support because content with measurable demand has higher observed traffic and a less negative trend. Staleness and engagement are useful as supporting signals, but their relationships are mixed and should not be treated as standalone rules.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.